# Preprocess the data & save

Input schema:  
`[token, date, open, high, low, close, volume]`


In [2]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('../data/crypto_data_old.csv')
df.sample(5)

,date,open,high,low,close,volume,token
693535,2021-10-06 00:00:00,0.4034,0.4084,0.395,0.3981,2665933,HBAR
648602,2023-09-12 16:00:00,1603.8400,1622.0000,1596.340,1598.9400,41562347,ETH
979945,2020-11-07 22:00:00,0.6060,0.6120,0.592,0.6100,206544,SUSHI
214479,2021-11-22 20:00:00,19.9100,20.1600,19.860,20.0100,111967,BAL
231739,2019-12-21 12:00:00,187.8200,187.9300,186.880,187.4300,80883,BCH


In [4]:
df['date'] = pd.to_datetime(df['date'])

In [5]:
print(f'Min: {df['date'].min()}\nMax: {df['date'].max()}')

Min: 2017-08-17 04:00:00
Max: 2023-10-19 23:00:00


## token extraction (BTC, ETH)

In [ ]:
# Feature extraction; we do not intend to trade these 
# symbols because the market is too efficient.
btc = df[df['token'] == 'BTC']
eth = df[df['token'] == 'ETH']

# Keep only close price for calculations
to_drop = ['open', 'high', 'low', 'volume', 'token']
btc = btc.drop(columns= to_drop).rename(columns={'close': 'btc_close'})
eth = eth.drop(columns= to_drop).rename(columns={'close': 'eth_close'})

btc.info()

<class 'pandas.core.frame.DataFrame'>
Index: 54116 entries, 317303 to 371418
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       54116 non-null  datetime64[ns]
 1   btc_close  54116 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 1.2 MB


In [39]:
print(len(df['token'].unique()))
df = df[~df['token'].isin({'ETH', 'BTC'})]
print(len(df['token'].unique()))

32
30


## next_close

In [40]:
# Create the target feature
df['next_close'] = df.groupby('token')['close'].shift(-1)
df.head()

,date,open,high,low,close,volume,token,next_close
0,2020-12-25 05:00:00,0.2000,3.0885,0.2000,2.5826,35530516,1INCH,2.5059
1,2020-12-25 06:00:00,2.5824,2.6900,2.2249,2.5059,22440875,1INCH,2.6237
2,2020-12-25 07:00:00,2.5152,2.8870,2.3609,2.6237,21300426,1INCH,2.6134
3,2020-12-25 08:00:00,2.6318,2.8247,2.4650,2.6134,17491813,1INCH,2.6365
4,2020-12-25 09:00:00,2.6104,2.7498,2.5629,2.6365,9919400,1INCH,2.7667


In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1011829 entries, 0 to 1120060
Data columns (total 8 columns):
 #   Column      Non-Null Count    Dtype         
---  ------      --------------    -----         
 0   date        1011829 non-null  datetime64[ns]
 1   open        1011829 non-null  float64       
 2   high        1011829 non-null  float64       
 3   low         1011829 non-null  float64       
 4   close       1011829 non-null  float64       
 5   volume      1011829 non-null  int64         
 6   token       1011829 non-null  object        
 7   next_close  1011799 non-null  float64       
dtypes: datetime64[ns](1), float64(5), int64(1), object(1)
memory usage: 69.5+ MB


## OHLCV_log (log returns)

In [ ]:
log_returns = lambda s: np.log(s / s.shift(1))  # noqa: E731
df['open_log'] = df.groupby('token')['open'].transform(log_returns)
df['high_log'] = df.groupby('token')['high'].transform(log_returns)
df['low_log'] = df.groupby('token')['low'].transform(log_returns)
df['close_log'] = df.groupby('token')['close'].transform(log_returns)
df['next_close_log'] = df.groupby('token')['next_close'].transform(log_returns)
df['volume_log'] = df.groupby('token')['volume'].transform(lambda x: np.log(1 + x))

In [43]:
df.head()

,date,open,high,low,close,volume,token,next_close,open_log,high_log,low_log,close_log,next_close_log,volume_log
0,2020-12-25 05:00:00,0.2000,3.0885,0.2000,2.5826,35530516,1INCH,2.5059,NaN,NaN,NaN,NaN,NaN,17.385903
1,2020-12-25 06:00:00,2.5824,2.6900,2.2249,2.5059,22440875,1INCH,2.6237,2.558157,-0.138144,2.409150,-0.030149,0.045938,16.926395
2,2020-12-25 07:00:00,2.5152,2.8870,2.3609,2.6237,21300426,1INCH,2.6134,-0.026367,0.070677,0.059331,0.045938,-0.003933,16.874238
3,2020-12-25 08:00:00,2.6318,2.8247,2.4650,2.6134,17491813,1INCH,2.6365,0.045316,-0.021816,0.043149,-0.003933,0.008800,16.677244
4,2020-12-25 09:00:00,2.6104,2.7498,2.5629,2.6365,9919400,1INCH,2.7667,-0.008165,-0.026874,0.038948,0.008800,0.048203,16.110003


## add year, eth, btc

In [ ]:
# Normalize year to [0, 1] scale for every 10 years
# This allows the model to extrapolate to future years naturally
df['year'] = (df['date'].dt.year - 2010) / 10

In [45]:
df = df.merge(btc, on='date', how='inner')
df = df.merge(eth, on='date', how='inner')

In [46]:
df['eth_btc_ratio'] = df['eth_close'] / df['btc_close']

In [47]:
df['btc_close_log'] = df.groupby('token')['btc_close'].transform(log_returns)
df['eth_close_log'] = df.groupby('token')['eth_close'].transform(log_returns)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1011829 entries, 0 to 1011828
Data columns (total 20 columns):
 #   Column          Non-Null Count    Dtype         
---  ------          --------------    -----         
 0   date            1011829 non-null  datetime64[ns]
 1   open            1011829 non-null  float64       
 2   high            1011829 non-null  float64       
 3   low             1011829 non-null  float64       
 4   close           1011829 non-null  float64       
 5   volume          1011829 non-null  int64         
 6   token           1011829 non-null  object        
 7   next_close      1011799 non-null  float64       
 8   open_log        1011799 non-null  float64       
 9   high_log        1011799 non-null  float64       
 10  low_log         1011799 non-null  float64       
 11  close_log       1011799 non-null  float64       
 12  next_close_log  1011769 non-null  float64       
 13  volume_log      1011829 non-null  float64       
 14  year            10

## drop null-containing rows

In [48]:
# drop null rows (next_close_log has the most)
df = df.dropna(subset=['next_close_log'])

## time_idx

In [49]:
# check to make sure everything is non-null
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1011769 entries, 1 to 1011827
Data columns (total 20 columns):
 #   Column          Non-Null Count    Dtype         
---  ------          --------------    -----         
 0   date            1011769 non-null  datetime64[ns]
 1   open            1011769 non-null  float64       
 2   high            1011769 non-null  float64       
 3   low             1011769 non-null  float64       
 4   close           1011769 non-null  float64       
 5   volume          1011769 non-null  int64         
 6   token           1011769 non-null  object        
 7   next_close      1011769 non-null  float64       
 8   open_log        1011769 non-null  float64       
 9   high_log        1011769 non-null  float64       
 10  low_log         1011769 non-null  float64       
 11  close_log       1011769 non-null  float64       
 12  next_close_log  1011769 non-null  float64       
 13  volume_log      1011769 non-null  float64       
 14  year            1011769

In [50]:
df['time_idx'] = df.groupby('token').cumcount()

# Cyclical features

In [51]:
PI = np.pi
df.head(5)

,date,open,high,low,close,volume,token,next_close,open_log,high_log,...,close_log,next_close_log,volume_log,year,btc_close,eth_close,eth_btc_ratio,btc_close_log,eth_close_log,time_idx
1,2020-12-25 06:00:00,2.5824,2.6900,2.2249,2.5059,22440875,1INCH,2.6237,2.558157,-0.138144,...,-0.030149,0.045938,16.926395,2020,23625.46,613.93,0.025986,0.001111,0.007636,0
2,2020-12-25 07:00:00,2.5152,2.8870,2.3609,2.6237,21300426,1INCH,2.6134,-0.026367,0.070677,...,0.045938,-0.003933,16.874238,2020,23649.82,612.92,0.025916,0.001031,-0.001646,1
3,2020-12-25 08:00:00,2.6318,2.8247,2.4650,2.6134,17491813,1INCH,2.6365,0.045316,-0.021816,...,-0.003933,0.008800,16.677244,2020,23677.44,617.01,0.026059,0.001167,0.006651,2
4,2020-12-25 09:00:00,2.6104,2.7498,2.5629,2.6365,9919400,1INCH,2.7667,-0.008165,-0.026874,...,0.008800,0.048203,16.110003,2020,23934.52,623.46,0.026049,0.010799,0.010399,3
5,2020-12-25 10:00:00,2.6365,2.9762,2.6345,2.7667,22372442,1INCH,2.6476,0.009949,0.079119,...,0.048203,-0.044002,16.923341,2020,23975.26,625.84,0.026104,0.001701,0.003810,4


## hour

In [ ]:
hour = lambda h: (2 * PI * h) / 24  # noqa: E731

df['hour_sin'] = np.sin(hour(df['date'].dt.hour))
df['hour_cos'] = np.cos(hour(df['date'].dt.hour))

## weekday

In [ ]:
weekday = lambda w: (2 * PI * w) / 7  # noqa: E731

df['weekday_sin'] = np.sin(weekday(df['date'].dt.dayofweek))
df['weekday_cos'] = np.cos(weekday(df['date'].dt.dayofweek))

## month

In [ ]:
month = lambda m: (2 * PI * m) / 12  # noqa: E731

df['month_sin'] = np.sin(month(df['date'].dt.month))
df['month_cos'] = np.cos(month(df['date'].dt.month))

## normalized day

In [ ]:
norm_day = lambda nd: (2 * PI * nd)  # noqa: E731

df['norm_day_sin'] = np.sin(norm_day(df['date'].dt.day / df['date'].dt.daysinmonth))
df['norm_day_cos'] = np.cos(norm_day(df['date'].dt.day / df['date'].dt.daysinmonth))

## drop cols used for feature extraction

In [ ]:
# drop columns used for feature extraction
to_drop = ['open', 'high', 'low', 'close', 'volume', 'next_close', 'btc_close', 'eth_close',
           'date']
df = df.drop(columns= to_drop)
df.columns

Index(['token', 'open_log', 'high_log', 'low_log', 'close_log',
       'next_close_log', 'volume_log', 'year', 'eth_btc_ratio',
       'btc_close_log', 'eth_close_log', 'time_idx', 'hour_sin', 'hour_cos',
       'weekday_sin', 'weekday_cos', 'month_sin', 'month_cos', 'norm_day_sin',
       'norm_day_cos'],
      dtype='object')

In [ ]:
col_order = ['time_idx', 'token', 'open_log', 'high_log', 'low_log', 'close_log', 'volume_log', 
             'btc_close_log', 'eth_close_log', 'eth_btc_ratio', 'hour_sin', 'hour_cos', 
             'norm_day_sin', 'norm_day_cos', 'weekday_sin', 'weekday_cos', 'month_sin', 
             'month_cos', 'year', 'next_close_log']
df = df.reindex(columns=col_order)

In [71]:
df.head(10)

,time_idx,token,open_log,high_log,low_log,close_log,volume_log,btc_close_log,eth_close_log,eth_btc_ratio,hour_sin,hour_cos,norm_day_sin,norm_day_cos,weekday_sin,weekday_cos,month_sin,month_cos,year,next_close_log
1,0,1INCH,2.558157,-0.138144,2.409150,-0.030149,16.926395,0.001111,0.007636,0.025986,1.000000e+00,6.123234e-17,-0.937752,0.347305,-0.433884,-0.900969,-2.449294e-16,1.0,2020,0.045938
2,1,1INCH,-0.026367,0.070677,0.059331,0.045938,16.874238,0.001031,-0.001646,0.025916,9.659258e-01,-2.588190e-01,-0.937752,0.347305,-0.433884,-0.900969,-2.449294e-16,1.0,2020,-0.003933
3,2,1INCH,0.045316,-0.021816,0.043149,-0.003933,16.677244,0.001167,0.006651,0.026059,8.660254e-01,-5.000000e-01,-0.937752,0.347305,-0.433884,-0.900969,-2.449294e-16,1.0,2020,0.008800
4,3,1INCH,-0.008165,-0.026874,0.038948,0.008800,16.110003,0.010799,0.010399,0.026049,7.071068e-01,-7.071068e-01,-0.937752,0.347305,-0.433884,-0.900969,-2.449294e-16,1.0,2020,0.048203
5,4,1INCH,0.009949,0.079119,0.027554,0.048203,16.923341,0.001701,0.003810,0.026104,5.000000e-01,-8.660254e-01,-0.937752,0.347305,-0.433884,-0.900969,-2.449294e-16,1.0,2020,-0.044002
6,5,1INCH,0.048167,-0.034943,-0.036529,-0.044002,16.590711,0.024683,0.005895,0.025618,2.588190e-01,-9.659258e-01,-0.937752,0.347305,-0.433884,-0.900969,-2.449294e-16,1.0,2020,-0.095937
7,6,1INCH,-0.043059,-0.026085,-0.068473,-0.095937,16.407365,-0.009298,-0.006838,0.025681,1.224647e-16,-1.000000e+00,-0.937752,0.347305,-0.433884,-0.900969,-2.449294e-16,1.0,2020,-0.040469
8,7,1INCH,-0.096428,-0.113209,-0.057171,-0.040469,16.590542,0.006266,0.004898,0.025646,-2.588190e-01,-9.659258e-01,-0.937752,0.347305,-0.433884,-0.900969,-2.449294e-16,1.0,2020,-0.118384
9,8,1INCH,-0.040884,-0.074844,-0.164667,-0.118384,16.925938,-0.019507,-0.015235,0.025755,-5.000000e-01,-8.660254e-01,-0.937752,0.347305,-0.433884,-0.900969,-2.449294e-16,1.0,2020,-0.085307
10,9,1INCH,-0.116826,-0.073730,-0.058298,-0.085307,16.732513,-0.000066,-0.008765,0.025532,-7.071068e-01,-7.071068e-01,-0.937752,0.347305,-0.433884,-0.900969,-2.449294e-16,1.0,2020,0.012394


## Save after preprocessing done
(This takes awhile so it's commented out)

In [ ]:
# df.to_csv('../data/preprocessed/crypto_data_proc.csv', index= False)